## 1. Original data loading

The original implementation expects MATLAB files containing three variables:

- `u_data`: branch network input.
- `x_data`: trunk network input (evaluation coordinates).
- `v_data`: target solution.

The corresponding loading function is shown below.

In [ ]:
def load_single_train(dataname):

    d = loadmat(dataname)

    u_data = d["u_data"]
    x_data = d["x_data"]
    v_data = d["v_data"]

    return u_data, x_data, v_data

## 2. Modified data loading

The bursting dataset has a different structure. Instead of the variables
used in the original implementation, each `.mat` file contains:

- `X`: branch input.
- `time`: temporal coordinates.
- `V`: membrane potential.
- `label`: burst class.
- `mean` and `std`: normalization statistics.

Therefore, the loading functions were adapted as follows.

In [ ]:
def load_single_train(dataname):

    data = loadmat(dataname)

    X = data["X"]
    time = data["time"].squeeze()
    V = data["V"]
    labels = data["label"].squeeze()
    mean = float(data["mean"].squeeze())
    std = float(data["std"].squeeze())

    return X, time, V, labels, mean, std

## 3. Main modifications

The following table summarizes the differences between the original
implementation and the adapted version.

| Original implementation | Modified implementation | Reason |
|------------------------|-------------------------|--------|
| `u_data` | `X` | New dataset variable name. |
| `x_data` | `time` | Time vector used as trunk input. |
| `v_data` | `V` | Membrane potential variable. |
| Not available | `label` | Added burst labels for supervised learning. |
| Not available | `mean`, `std` | Saved normalization statistics. |



## 4. Conversion to PyTorch tensors

After loading the MATLAB arrays, they are converted into PyTorch tensors
to enable training with DeepONet.

In [ ]:
X = torch.tensor(X, dtype=torch.float32)
time = torch.tensor(time, dtype=torch.float32).unsqueeze(1)
V = torch.tensor(V, dtype=torch.float32)
labels = torch.tensor(labels, dtype=torch.long)

The conversion performs the following operations:

- `float32` is used for all continuous variables, as it is the standard
  data type for neural network training.
- `unsqueeze(1)` transforms the time vector from shape `(N,)` to `(N,1)`,
  which is required by the trunk network.
- `labels` are converted to `torch.long`, since class labels must be stored
  as integer tensors for classification tasks.